# Day 10 — Flight Operations Data Analysis

This notebook analyzes the Flight Operations dataset using Pandas. The workflow covers loading the data, understanding its structure, checking data quality, selecting and filtering records, sorting, grouping and aggregation, transformations, and drawing practical observations.

**Dataset:** `Day10_Flight_Operations_Dataset.csv`  
**Tools:** Python, Pandas, Google Colab


## 1. Import Libraries and Load the Dataset

The dataset is loaded into a DataFrame so the flight records can be inspected and analyzed using Pandas operations.


In [ ]:
import pandas as pd
import numpy as np

file_path = 'Day10_Flight_Operations_Dataset.csv'

try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    from google.colab import files
    uploaded = files.upload()
    file_path = next(iter(uploaded))
    df = pd.read_csv(file_path)

df.head()


## 2. Understand the Dataset

Inspect the number of records, columns, data types, and a sample of the data before starting the detailed analysis.


In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

print('\nData types and non-null counts:')
df.info()

display(df.head(10))


## 3. Descriptive Statistics

Summary statistics show the typical values and ranges of important numerical fields such as passengers, ticket price, delays, baggage, and satisfaction.


In [ ]:
df.describe().T


## 4. Data Quality Check

Check missing values and duplicate rows. The delay column contains missing values, so those records are handled before calculations that depend on delay.


In [ ]:
print('Missing values by column:')
display(df.isnull().sum().to_frame('Missing_Count'))

print('Duplicate rows:', df.duplicated().sum())

df['Flight_Date'] = pd.to_datetime(df['Flight_Date'])

delay_median = df['Delay_Minutes'].median()
df['Delay_Minutes'] = df['Delay_Minutes'].fillna(delay_median)

print('Median delay used for missing values:', delay_median)
print('Remaining missing values:', df.isnull().sum().sum())


## 5. Selecting Relevant Data

Use column selection and row selection to focus on information needed for specific questions.


In [ ]:
display(df[['Flight_ID', 'Airline', 'Origin', 'Destination',
                  'Passengers', 'Average_Ticket_Price', 'Flight_Status']].head(10))

high_passenger_flights = df[df['Passengers'] >= 180]
print('Flights with at least 180 passengers:', len(high_passenger_flights))
display(high_passenger_flights[['Flight_ID', 'Airline', 'Passengers',
                                'Seat_Capacity', 'Flight_Status']].head(10))


## 6. Filtering Records

Filtering can answer operational questions such as which flights were delayed, which flights had high ticket prices, and which flights combined delays with lower satisfaction.


In [ ]:
delayed = df[df['Flight_Status'] == 'Delayed']
high_price = df[df['Average_Ticket_Price'] > 10000]
delay_low_sat = df[(df['Flight_Status'] == 'Delayed') &
                   (df['Passenger_Satisfaction'] <= 3)]

print('Delayed flights:', len(delayed))
print('Flights with ticket price above 10,000:', len(high_price))
print('Delayed flights with satisfaction <= 3:', len(delay_low_sat))

display(delay_low_sat[['Flight_ID', 'Airline', 'Delay_Minutes',
                       'Weather', 'Passenger_Satisfaction']].head(10))


## 7. Sorting Records

Sort the records to identify flights with the highest ticket prices, longest delays, and largest passenger counts.


In [ ]:
print('Top 10 ticket prices:')
display(df.sort_values('Average_Ticket_Price', ascending=False)
        [['Flight_ID', 'Airline', 'Travel_Class', 'Average_Ticket_Price']].head(10))

print('Top 10 delays:')
display(df.sort_values('Delay_Minutes', ascending=False)
        [['Flight_ID', 'Airline', 'Weather', 'Delay_Minutes', 'Flight_Status']].head(10))

print('Top 10 passenger counts:')
display(df.sort_values('Passengers', ascending=False)
        [['Flight_ID', 'Airline', 'Passengers', 'Seat_Capacity']].head(10))


## 8. Data Transformations

Create additional columns that make the operational analysis more useful. Occupancy rate estimates how full each flight was, while estimated revenue uses passengers and average ticket price.


In [ ]:
df['Occupancy_Rate'] = (df['Passengers'] / df['Seat_Capacity']) * 100
df['Estimated_Revenue'] = df['Passengers'] * df['Average_Ticket_Price']
df['Month'] = df['Flight_Date'].dt.month_name()
df['Day_of_Week'] = df['Flight_Date'].dt.day_name()

df['Delay_Category'] = np.select(
    [
        df['Delay_Minutes'].eq(0),
        df['Delay_Minutes'].le(15),
        df['Delay_Minutes'].le(60),
        df['Delay_Minutes'].gt(60)
    ],
    ['No Delay', 'Short Delay', 'Moderate Delay', 'Long Delay'],
    default='Unknown'
)

display(df[['Flight_ID', 'Passengers', 'Seat_Capacity', 'Occupancy_Rate',
            'Estimated_Revenue', 'Month', 'Day_of_Week',
            'Delay_Category']].head(10))


## 9. Airline-wise Analysis

Compare airlines using flight count, total passengers, estimated revenue, average delay, and passenger satisfaction.


In [ ]:
airline_analysis = df.groupby('Airline').agg(
    Flights=('Flight_ID', 'count'),
    Total_Passengers=('Passengers', 'sum'),
    Estimated_Revenue=('Estimated_Revenue', 'sum'),
    Average_Delay=('Delay_Minutes', 'mean'),
    Average_Satisfaction=('Passenger_Satisfaction', 'mean')
).sort_values('Estimated_Revenue', ascending=False)

airline_analysis


## 10. Flight Status Analysis

Compare cancelled, delayed, and on-time flights to understand operational distribution and passenger experience.


In [ ]:
status_analysis = df.groupby('Flight_Status').agg(
    Flights=('Flight_ID', 'count'),
    Total_Passengers=('Passengers', 'sum'),
    Average_Delay=('Delay_Minutes', 'mean'),
    Average_Satisfaction=('Passenger_Satisfaction', 'mean')
).sort_values('Flights', ascending=False)

status_analysis


## 11. Travel Class Analysis

Analyze passenger volume, average ticket price, estimated revenue, and satisfaction across travel classes.


In [ ]:
class_analysis = df.groupby('Travel_Class').agg(
    Flights=('Flight_ID', 'count'),
    Total_Passengers=('Passengers', 'sum'),
    Average_Ticket_Price=('Average_Ticket_Price', 'mean'),
    Estimated_Revenue=('Estimated_Revenue', 'sum'),
    Average_Satisfaction=('Passenger_Satisfaction', 'mean')
).sort_values('Estimated_Revenue', ascending=False)

class_analysis


## 12. Weather and Delay Analysis

Compare weather conditions with delay levels and satisfaction to identify possible operational patterns.


In [ ]:
weather_analysis = df.groupby('Weather').agg(
    Flights=('Flight_ID', 'count'),
    Average_Delay=('Delay_Minutes', 'mean'),
    Delayed_Flights=('Flight_Status', lambda x: (x == 'Delayed').sum()),
    Average_Satisfaction=('Passenger_Satisfaction', 'mean')
)

weather_analysis['Delayed_Rate_%'] = (
    weather_analysis['Delayed_Flights'] / weather_analysis['Flights'] * 100
)

weather_analysis.sort_values('Delayed_Rate_%', ascending=False)


## 13. Booking Channel Analysis

Compare booking channels by passenger volume, average ticket price, and satisfaction.


In [ ]:
booking_analysis = df.groupby('Booking_Channel').agg(
    Flights=('Flight_ID', 'count'),
    Total_Passengers=('Passengers', 'sum'),
    Average_Ticket_Price=('Average_Ticket_Price', 'mean'),
    Average_Satisfaction=('Passenger_Satisfaction', 'mean')
).sort_values('Total_Passengers', ascending=False)

booking_analysis


## 14. Origin and Destination Analysis

Identify the busiest origin and destination locations using grouped passenger counts.


In [ ]:
origin_analysis = df.groupby('Origin').agg(
    Flights=('Flight_ID', 'count'),
    Total_Passengers=('Passengers', 'sum')
).sort_values('Total_Passengers', ascending=False)

destination_analysis = df.groupby('Destination').agg(
    Flights=('Flight_ID', 'count'),
    Total_Passengers=('Passengers', 'sum')
).sort_values('Total_Passengers', ascending=False)

print('Top origins:')
display(origin_analysis.head(5))

print('Top destinations:')
display(destination_analysis.head(5))


## 15. Overall Performance Metrics

Calculate overall indicators to summarize the dataset.


In [ ]:
summary = pd.Series({
    'Total Flights': len(df),
    'Total Passengers': df['Passengers'].sum(),
    'Estimated Revenue': df['Estimated_Revenue'].sum(),
    'Average Ticket Price': df['Average_Ticket_Price'].mean(),
    'Average Delay (minutes)': df['Delay_Minutes'].mean(),
    'Average Passenger Satisfaction': df['Passenger_Satisfaction'].mean(),
    'Average Occupancy Rate (%)': df['Occupancy_Rate'].mean()
})

summary


## 16. Key Observations

1. The dataset contains **180 flight records** across **6 airlines**, allowing airline-level comparisons.
2. **Vistara** generated the highest estimated revenue in this dataset, while **Air India** recorded the highest average passenger satisfaction among the airlines.
3. **114 flights were on time**, compared with 59 delayed and 7 cancelled flights.
4. Delayed flights had an average delay of about **50 minutes** and lower average satisfaction than on-time flights.
5. **Economy class** carried the largest number of passengers and generated the highest total estimated revenue.
6. **Premium Economy** had the highest average passenger satisfaction among the three travel classes.
7. **Delhi** was the busiest origin by passenger volume, while **Storm** weather had a relatively high delayed-flight rate.
8. Several flights have passenger counts above recorded seat capacity, producing occupancy rates above 100%. This should be investigated as a data-quality issue before using occupancy for operational decisions.

These observations describe patterns in the supplied sample and should not be treated as general airline-industry conclusions.


## 17. Conclusion

The analysis used Pandas selection, filtering, sorting, grouping, aggregation, datetime operations, missing-value handling, and derived columns to examine flight operations. The results highlight differences between airlines, travel classes, booking channels, weather conditions, and flight statuses, while also identifying a data-quality issue in passenger and seat-capacity values.
